# Predict Aging clock from DNA methylation data

In [1]:
## Importing all the necessary libraries
import pandas as pd
from preprocess._preprocess import df_to_adata
from preprocess._preprocess_utils import *
from models.model import MyModel
from predict.predict import Predictor

## Data Preprocessing

In [2]:
df = pd.read_pickle("../tests/GSE139307_half.pkl")
print(df.columns)

Index(['dataset', 'tissue_type', 'age', 'gender', 'cg00000029', 'cg00000108',
       'cg00000109', 'cg00000165', 'cg00000236', 'cg00000289',
       ...
       'ch.X.93511680F', 'ch.X.938089F', 'ch.X.94051109R', 'ch.X.94260649R',
       'ch.X.967194F', 'ch.X.97129969R', 'ch.X.97133160R', 'ch.X.97651759F',
       'ch.X.97737721F', 'ch.X.98007042R'],
      dtype='object', length=485516)


In [3]:
df['female'] = (df['gender'] == 'F').astype(int)

In [4]:
import pyaging as pya 
df = pya.pp.epicv2_probe_aggregation(df)

|-----> 🏗️ Starting epicv2_probe_aggregation function
|-----> ⚙️ Looking for duplicated probes started
|-----------> in progress: 100.0000%
|-----------> There are no duplicated probes. Returning original data
|-----> 🎉 Done! [16.6989s]


In [5]:
adata = df_to_adata(df, metadata_cols=['dataset', 'tissue_type',  'gender'], imputer_strategy='knn')

Number of observations: 18
Number of features: 485513
Total missing values: 331
Percentage of missing values: 0.00%


In [6]:
# Moreover, it is important to note that some probes are duplicated in the EPICv2 array, following the format cg#########_BC11 and cg#########_TC11 for the opposite strands. Given that at this moment most clocks have not been trained with EPICv2 data directly, it is recommended to average these probes. This is particularly the case for DunedinPACE, from which some clock probes were duplicated in the update from EPICv1. To remedy this issue, simply use the following function to aggregate any duplicated probes that may be present.
# らしい


In [7]:
adata

AnnData object with n_obs × n_vars = 18 × 485513
    obs: 'dataset', 'tissue_type', 'gender'
    var: 'percent_na'
    uns: 'imputer_strategy'
    layers: 'X_original', 'X_imputed'

## Setup Prediction model

In [8]:
features = adata.var.index.values
clock_name = "horvath2013"
batch_size = 1024
# model = MyModel(len(features), 1, clock_name, features)

## Run prediction

In [9]:
print(features)

['age' 'cg00000029' 'cg00000108' ... 'ch.X.97737721F' 'ch.X.98007042R'
 'female']


In [10]:
import torch
from anndata.experimental.pytorch import AnnLoader


weights_path = 'pyaging_data/horvath2013.pt'

model = torch.load(weights_path, weights_only=False)
print(model)

# Horvath2013(
#   (base_model): LinearModel(
#     (linear): Linear(in_features=353, out_features=1, bias=True)
#   )
# )


print(model.metadata)

# {'clock_name': 'horvath2013',
#  'data_type': 'methylation',
#  'species': 'Homo sapiens',
#  'year': 2013,
#  'approved_by_author': '⌛',
#  'citation': 'Horvath, Steve. "DNA methylation age of human tissues and cell types." Genome biology 14.10 (2013): 1-20.',
#  'doi': 'https://doi.org/10.1186/gb-2013-14-10-r115',
#  'notes': None,
#  'research_only': None,
#  'version': None}

Horvath2013(
  (base_model): LinearModel(
    (linear): Linear(in_features=353, out_features=1, bias=True)
  )
)
{'clock_name': 'horvath2013', 'data_type': 'methylation', 'species': 'Homo sapiens', 'year': 2013, 'approved_by_author': '⌛', 'citation': 'Horvath, Steve. "DNA methylation age of human tissues and cell types." Genome biology 14.10 (2013): 1-20.', 'doi': 'https://doi.org/10.1186/gb-2013-14-10-r115', 'notes': None, 'research_only': None, 'version': None}


In [11]:

# Preallocate the data matrix
adata.obsm[f"X_{model.metadata['clock_name']}"] = (
    np.empty((adata.n_obs, len(model.features)), order="F")
)

In [12]:


# Find indices of matching features in adata.var_names
feature_indices = {feature: i for i, feature in enumerate(adata.var_names)}
model_feature_indices = np.array([feature_indices.get(feature, -1) for feature in model.features])

# Identify missing features
missing_features_mask = model_feature_indices == -1
missing_features = np.array(model.features)[missing_features_mask].tolist()

# Assign values for existing features
existing_features_mask = ~missing_features_mask
existing_features_indices = model_feature_indices[existing_features_mask]
adata.obsm[f"X_{model.metadata['clock_name']}"][:, existing_features_mask] = adata.X[:, existing_features_indices]

# Handle missing features
adata.obsm[f"X_{model.metadata['clock_name']}"][:, missing_features_mask] = (
    np.array(model.reference_values)[missing_features_mask] if model.reference_values is not None else 0
)

# Calculate missing features statistics
num_missing_features = len(missing_features)
percent_missing = 100 * num_missing_features / len(model.features)

# Add missing features and percent missing values to the clock
adata.uns[f"{model.metadata['clock_name']}_percent_na"] = percent_missing
adata.uns[f"{model.metadata['clock_name']}_missing_features"] = missing_features

# Raises error if there are no features in the data
if percent_missing == 100:
    logger.error(
        f"Every single feature out of {len(model.features)} features "
        f"is missing. Please double check the features in the adata object"
        f" actually contain the clock features such as {missing_features[:np.min([3, num_missing_features])]}, etc.",
        indent_level=3,
    )
    raise NameError


In [13]:
use_cuda = torch.cuda.is_available()
dataloader = AnnLoader(adata, batch_size=batch_size, use_cuda=use_cuda)

# Use the AnnLoader for batched prediction
predictions = []
with torch.inference_mode():
    for batch in dataloader:
        batch_pred = model(batch.obsm[f"X_{model.metadata['clock_name']}"])
        predictions.append(batch_pred)
# Concatenate all batch predictions
predictions = torch.cat(predictions)

In [14]:
print(predictions)

# チュートリアルと一致
# tensor([[22.9123],
#         [29.2688],
#         [28.9278],
#         [35.5918],
#         [30.6931],
#         [25.2088],
#         [25.4727],
#         [25.2818],
#         [21.6082],
#         [26.5560],
#         [23.7388],
#         [21.4079],
#         [25.3795],
#         [26.9945],
#         [29.8154],
#         [29.3709],
#         [27.1390],
#         [23.6596]], dtype=torch.float64)


tensor([[22.9123],
        [29.2688],
        [28.9278],
        [35.5918],
        [30.6931],
        [25.2088],
        [25.4727],
        [25.2818],
        [21.6082],
        [26.5560],
        [23.7388],
        [21.4079],
        [25.3795],
        [26.9945],
        [29.8154],
        [29.3709],
        [27.1390],
        [23.6596]], dtype=torch.float64)


In [15]:
def postpro(y):
    adult_age = 20
    x = y.clone()
    x -= adult_age
    result = np.where(x < 0, (1 + adult_age) * np.exp(x) - 1, (1 + adult_age) * x + adult_age)
    return result

In [16]:
predictions[:10]

tensor([[22.9123],
        [29.2688],
        [28.9278],
        [35.5918],
        [30.6931],
        [25.2088],
        [25.4727],
        [25.2818],
        [21.6082],
        [26.5560]], dtype=torch.float64)

In [17]:
age = postpro(predictions[:10])
print(age)

[[ 81.15929866]
 [214.64567335]
 [207.4836107 ]
 [347.42855764]
 [244.55523137]
 [129.38517387]
 [134.92683348]
 [130.91731922]
 [ 53.77175087]
 [157.6758673 ]]


In [18]:
df['age'][:10]

GSM4137727    66.0
GSM4137728    84.0
GSM4137729    69.0
GSM4137730    82.0
GSM4137731    69.0
GSM4137732    84.0
GSM4137733    65.0
GSM4137734    84.0
GSM4137735    67.0
GSM4137736    84.0
Name: age, dtype: float64

In [ ]:
# データさえあれば、これだけでできる。手法と、データは分離して管理しておいた方がいいね。データベースの管理の方が重要そう。

In [19]:
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': [0.1, 0.5, 1.0, 5.0, 10.0],  # 正則化の強さ
    'l1_ratio': [0.1, 0.5, 0.7, 0.9, 1.0]  # L1ペナルティとL2ペナルティのバランス
}

# ElasticNetモデルを作成
model = ElasticNet()

# グリッドサーチの設定
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')

grid_search.fit(adata.layers['X_imputed'][:10], df['age'][:10])
print("Best parameters found: ", grid_search.best_params_)



best_model = grid_search.best_estimator_

# 残りのデータに対して予測
predictions = best_model.predict(adata.layers['X_imputed'][10:])
print(predictions)
print( df['age'][10:])

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 0.1}
[72.00493254 67.01218628 85.98462208 69.00928478 67.01218628 69.00928478
 69.00928478 67.01218628]
GSM4137737    72.0
GSM4137738    67.0
GSM4137739    86.0
GSM4137740    69.0
GSM4137741    67.0
GSM4137742    69.0
GSM4137743    69.0
GSM4137744    67.0
Name: age, dtype: float64
